# Lab 08-10 · Timing, Metastability and Clock-Domain Crossing

Companion to chapter [08-10](../../chapters/08-10-timing-metastability-cdc.md). Handout: [README.md](README.md).

Cells marked **🔮 Predict** ask for a prediction *before* running.

In [1]:
import itertools
import math
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

YEAR = 365.25 * 24 * 3600

## Part 1 · Static timing analysis on a small netlist

Three registers launch signals at $t_{pcq}$ = 50 ps; six gates combine them; two registers capture them with $t_{\text{setup}}$ = 30 ps. Each gate lists its delay and its inputs. STA computes every arrival time in one pass in topological order; the cell then checks the answer by enumerating every path.

In [2]:
T_PCQ, T_SETUP = 50, 30
gates = {               # name: (delay in ps, inputs)
    "g1": (40, ["R1", "R2"]),
    "g2": (60, ["g1", "R3"]),
    "g3": (45, ["g2", "g1"]),
    "g4": (20, ["R3"]),
    "g5": (55, ["g4", "g2"]),
    "g6": (70, ["g3", "g5"]),
}
captures = {"R4": "g6", "R5": "g4"}
launch = ["R1", "R2", "R3"]

# one pass in topological order (the dictionary is already ordered so that inputs come first)
arrival = {r: T_PCQ for r in launch}
worst_in = {}
for g, (d, ins) in gates.items():
    src = max(ins, key=lambda n: arrival[n])
    arrival[g] = arrival[src] + d
    worst_in[g] = src

for reg, net in captures.items():
    path, n = [], net
    while n in gates:
        path.append(n)
        n = worst_in[n]
    path.append(n)
    print(f"{reg}: arrival {arrival[net]} ps, min period {arrival[net] + T_SETUP} ps, critical path {' -> '.join(reversed(path))}")
crit = max(arrival[n] for n in captures.values())
period_3ghz = 1e3 / 3
print(f"f_max = {1e3 / (crit + T_SETUP):.2f} GHz;  slack at 3 GHz = {period_3ghz - T_SETUP - crit:.1f} ps")


# brute force: enumerate every launch-to-capture path and its delay
def paths_to(n):
    if n in launch:
        return [[n]]
    d, ins = gates[n]
    return [p + [n] for i in ins for p in paths_to(i)]


all_paths = [p for net in captures.values() for p in paths_to(net)]
delays = [T_PCQ + sum(gates[g][0] for g in p[1:]) for p in all_paths]
print(f"{len(all_paths)} paths; the longest is {max(delays)} ps")
assert max(delays) == crit == 275

R4: arrival 275 ps, min period 305 ps, critical path R1 -> g1 -> g2 -> g5 -> g6
R5: arrival 70 ps, min period 100 ps, critical path R3 -> g4
f_max = 3.28 GHz;  slack at 3 GHz = 28.3 ps
10 paths; the longest is 275 ps


The topological pass visits each gate once, while the number of paths can grow exponentially with circuit depth (10 here, astronomically many in a real chip). That is why STA propagates arrival times instead of enumerating paths.

## Part 2 · Metastability, simulated

Model: a data transition arrives at an offset $\Delta t$ from the flip-flop's balance instant, uniformly over the clock period; the initial imbalance is proportional to $|\Delta t|$; and it grows as $e^{t/\tau}$. A sample whose imbalance starts at the full logic level resolves at once.

**🔮 Predict:** on a semilog plot, what shape does "fraction still unresolved after time $t$" have, and what are its slope and intercept?

In [3]:
rng = np.random.default_rng(810)
TAU, TW, TC, N = 20e-12, 20e-12, 1e-9, 10_000_000
dt = rng.uniform(-TC / 2, TC / 2, N)
ratio = np.abs(dt) / (TW / 2)                           # initial imbalance relative to a resolved level
t_res = np.where(ratio >= 1, 0.0, TAU * np.log(1 / np.maximum(ratio, 1e-300)))
t = np.arange(20e-12, 160e-12, 10e-12)
frac = np.array([(t_res > x).mean() for x in t])
slope, intercept = np.polyfit(t, np.log(frac), 1)
tau_fit, tw_fit = -1 / slope, np.exp(intercept) * TC
print(f"fitted tau = {tau_fit * 1e12:.2f} ps (true 20), fitted T_W = {tw_fit * 1e12:.2f} ps (true 20)")
assert abs(tau_fit - TAU) < 0.5e-12 and abs(tw_fit - TW) < 1e-12

fitted tau = 19.94 ps (true 20), fitted T_W = 20.09 ps (true 20)


A straight line on a semilog plot: the fraction unresolved after $t$ is $(T_W/T_c)e^{-t/\tau}$, exactly the window argument of chapter §3.4. Real measurements of $\tau$ and $T_W$ (Problem 08-10.14, and the test chips of §9) work the same way: they shorten the resolution time until failures become countable and fit the exponential.

## Part 3 · Synchronizer MTBF (the chapter's guided exercise)

**🔮 Predict:** with $\tau$ = 20 ps, $T_W$ = 20 ps, $t_{pcq} + t_{\text{setup}}$ = 80 ps and 200 MHz data, by what factor does a two-flip-flop synchronizer's MTBF change when its clock goes from 500 MHz to 1 GHz?

In [4]:
def mtbf_s(fc, tau=20e-12, tw=20e-12, fd=200e6, stages=2, overhead=80e-12):
    S = (stages - 1) * (1 / fc - overhead)
    return math.exp(S / tau) / (tw * fc * fd)


for fc in (250e6, 500e6, 1e9, 2e9):
    row = "  ".join(f"{mtbf_s(fc, stages=k) / YEAR:9.2e}" for k in (2, 3, 4))
    print(f"{fc / 1e6:5.0f} MHz   MTBF in years for 2 / 3 / 4 flip-flops: {row}")
ratio = mtbf_s(500e6) / mtbf_s(1e9)
print(f"\n500 MHz vs 1 GHz, two flip-flops: ratio {ratio:.3e} (10^{math.log10(ratio):.1f})")

  250 MHz   MTBF in years for 2 / 3 / 4 flip-flops:  4.19e+71  5.55e+156  7.35e+241
  500 MHz   MTBF in years for 2 / 3 / 4 flip-flops:  7.80e+27   3.84e+69  1.89e+111
 1000 MHz   MTBF in years for 2 / 3 / 4 flip-flops:  7.52e+05   7.14e+25   6.78e+45
 2000 MHz   MTBF in years for 2 / 3 / 4 flip-flops:  5.22e-06   6.89e+03   9.09e+12

500 MHz vs 1 GHz, two flip-flops: ratio 1.037e+22 (10^22.0)


Halving the clock frequency doubles the denominator's $F_C$ only once, but it also gives the first flip-flop 1000 ps more to resolve, 50 more time constants, which multiplies the MTBF by $e^{50} \approx 5 \times 10^{21}$. The ratio is $2e^{50} \approx 10^{22}$. At 2 GHz the same two flip-flops fail about every 165 seconds (5.2e-6 years), which is why fast domains need longer synchronizers.

In [5]:
def log10_mtbf_years(fc, stages, tau=20e-12, tw=20e-12, fd=200e6, overhead=80e-12):
    # computed in logarithms: e^(S/tau) overflows a float for slow clocks and long chains
    S = (stages - 1) * (1 / fc - overhead)
    return S / tau / math.log(10) - math.log10(tw * fc * fd) - math.log10(YEAR)


fs = np.linspace(500e6, 3e9, 200)
fig, ax = plt.subplots(figsize=(6.5, 3.6))
for k in (2, 3, 4):
    ax.plot(fs / 1e9, [log10_mtbf_years(f, k) for f in fs], label=f"{k} flip-flops")
ax.axhline(3, color="0.5", ls=":", lw=0.8)
ax.text(2.95, 5, "1000 years", ha="right", fontsize=8, color="0.4")
ax.set_xlabel("destination clock (GHz)")
ax.set_ylabel("log10 of MTBF in years")
ax.set_ylim(-10, 60)
ax.legend()
ax.set_title("Synchronizer MTBF against clock frequency (τ = 20 ps)")
plt.show()

## Part 4 · Sampling a counter mid-transition

For every transition of a 4-bit counter (including the wrap from 15 to 0), every bit that changes can be sampled old or new independently. Count the sampled values that are neither the old nor the new count.

In [6]:
def gray(x):
    return x ^ (x >> 1)


def impossible_samples(code):
    total = 0
    for c in range(16):
        a, b = code(c), code((c + 1) % 16)
        changed = [i for i in range(4) if ((a ^ b) >> i) & 1]
        seen = set()
        for pick in itertools.product((0, 1), repeat=len(changed)):
            v = a
            for i, p in zip(changed, pick):
                if p:
                    v ^= 1 << i
            seen.add(v)
        total += len(seen - {a, b})
    return total


print("impossible values over a full cycle: binary", impossible_samples(lambda x: x), "  Gray", impossible_samples(gray))
assert impossible_samples(gray) == 0

impossible values over a full cycle: binary 48   Gray 0


Binary counting produces 48 impossible values over one cycle of 16 transitions; Gray counting produces none, because every transition changes exactly one bit. This is the property the asynchronous FIFO in `cdc.v` relies on.

**Verilog.** Now run `cdc_tb.v` (`python tools/run_hdl.py` from the repository root). Remember what it can and cannot show: Icarus never produces a metastable value, so the testbench checks the crossing *logic* (Gray pointers, flags, toggles), not the probability of failure, which is what Parts 2 and 3 are for.